In [2]:
from copy import deepcopy
from collections import deque


# ============================================================
# 1. PROBLEM DEFINITION
# ============================================================

# What are we assigning?
VARIABLES = [
    "A",
    "B",
    "C",
]


# What values can each variable take?
DOMAINS = {
    "A": ["Red", "Green", "Blue"],
    "B": ["Red", "Green", "Blue"],
    "C": ["Red", "Green", "Blue"],
}


# Which variables have constraints with each other?
CONSTRAINTS = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
]


# ============================================================
# 2. BUILD NEIGHBORS
# ============================================================

def build_neighbors():
    neighbors = {
        variable: set()
        for variable in VARIABLES
    }

    for x, y in CONSTRAINTS:
        neighbors[x].add(y)
        neighbors[y].add(x)

    return neighbors


NEIGHBORS = build_neighbors()


# ============================================================
# 3. CONSTRAINT CHECK
# ============================================================

def is_consistent(variable, value, assignment):

    for neighbor in NEIGHBORS[variable]:

        if neighbor in assignment:

            # Most common CSP:
            # neighboring variables cannot have same value

            if assignment[neighbor] == value:
                return False

    return True


# ============================================================
# 4. BASIC BACKTRACKING
# ============================================================

def backtracking(assignment, domains):

    # Goal test
    if len(assignment) == len(VARIABLES):
        return assignment.copy()


    # Select variable
    variable = select_variable(
        assignment,
        domains
    )


    # Try every possible value
    for value in domains[variable]:

        # Check constraints
        if is_consistent(
            variable,
            value,
            assignment
        ):

            # Make decision
            assignment[variable] = value


            # Recursive search
            result = backtracking(
                assignment,
                domains
            )


            # Solution found
            if result is not None:
                return result


            # Undo decision
            del assignment[variable]


    # No value worked
    return None


# ============================================================
# 5. VARIABLE SELECTION
#    MRV + DEGREE HEURISTIC
# ============================================================

def select_variable(assignment, domains):

    # Only unassigned variables
    unassigned = [
        v for v in VARIABLES
        if v not in assignment
    ]


    # MRV + Degree
    return min(
        unassigned,
        key=lambda v: (
            len(domains[v]),
            -sum(
                1
                for n in NEIGHBORS[v]
                if n not in assignment
            )
        )
    )


# ============================================================
# 6. FORWARD CHECKING
# ============================================================

def forward_check(
    variable,
    value,
    assignment,
    domains
):

    new_domains = deepcopy(domains)


    # Assigned variable has only one value
    new_domains[variable] = [value]


    # Remove value from neighbors
    for neighbor in NEIGHBORS[variable]:

        if neighbor in assignment:
            continue


        if value in new_domains[neighbor]:

            new_domains[neighbor].remove(value)


        # Domain wipeout = failure
        if len(new_domains[neighbor]) == 0:
            return None


    return new_domains


# ============================================================
# 7. AC-3 : REVISE
# ============================================================

def revise(domains, x, y):

    revised = False


    for x_value in domains[x][:]:

        # Does x_value have at least one
        # compatible value in Y?

        supported = any(
            x_value != y_value
            for y_value in domains[y]
        )


        # No support -> remove
        if not supported:

            domains[x].remove(x_value)

            revised = True


    return revised


# ============================================================
# 8. AC-3
# ============================================================

def ac3(domains):

    queue = deque()


    # Put every arc into queue
    for x in VARIABLES:

        for y in NEIGHBORS[x]:

            queue.append((x, y))


    # Process queue
    while queue:

        x, y = queue.popleft()


        # Revise X using Y
        if revise(domains, x, y):


            # Empty domain = failure
            if len(domains[x]) == 0:
                return False


            # X changed,
            # so recheck its other neighbors
            for z in NEIGHBORS[x]:

                if z != y:

                    queue.append((z, x))


    return True


# ============================================================
# 9. IMPROVED BACKTRACKING
#    MRV + DEGREE + FC + AC-3
# ============================================================

def improved_backtracking(
    assignment,
    domains
):

    # Goal
    if len(assignment) == len(VARIABLES):
        return assignment.copy()


    # MRV + Degree
    variable = select_variable(
        assignment,
        domains
    )


    # Try values
    for value in domains[variable]:


        # Check current assignment
        if not is_consistent(
            variable,
            value,
            assignment
        ):
            continue


        # Forward checking
        new_domains = forward_check(
            variable,
            value,
            assignment,
            domains
        )


        # Failure
        if new_domains is None:
            continue


        # AC-3 propagation
        if not ac3(new_domains):
            continue


        # Assign
        assignment[variable] = value


        # Recursive search
        result = improved_backtracking(
            assignment,
            new_domains
        )


        # Success
        if result is not None:
            return result


        # Undo
        del assignment[variable]


    return None


# ============================================================
# 10. SOLVE
# ============================================================

def solve():

    domains = deepcopy(DOMAINS)

    assignment = {}


    # Optional AC-3 before search
    if not ac3(domains):

        print("No solution exists.")

        return None


    solution = improved_backtracking(
        assignment,
        domains
    )


    return solution


# ============================================================
# 11. MAIN
# ============================================================

if __name__ == "__main__":

    solution = solve()

    print("Solution:")

    print(solution)

Solution:
{'A': 'Red', 'B': 'Green', 'C': 'Blue'}


In [3]:
"""
Constraint Satisfaction Problem Lab
-----------------------------------

Problem:
    Color each region of a map using 4 colors so that neighboring
    regions have different colors.

1. Basic backtracking:
       - fixed variable ordering
       - filtering against assigned variables

2. Improved backtracking:
       - MRV variable ordering
       - forward checking
       - AC-3 constraint propagation

The program counts search nodes so that the amount of exploration
can be compared.
"""

from copy import deepcopy
from collections import deque


COLORS = ["Red", "Green", "Blue", "Yellow"]

EDGES = [
    ("A", "B"),
    ("A", "C"),
    ("A", "D"),
    ("B", "C"),
    ("B", "E"),
    ("B", "F"),
    ("C", "D"),
    ("C", "E"),
    ("C", "F"),
    ("D", "E"),
    ("D", "G"),
    ("E", "F"),
    ("E", "G"),
    ("E", "H"),
    ("F", "G"),
    ("F", "H"),
    ("G", "H"),
]

VARIABLES = sorted(set(x for edge in EDGES for x in edge))

DOMAINS = {
    variable: COLORS.copy()
    for variable in VARIABLES
}


def build_neighbors(edges):
    neighbors = {v: set() for v in VARIABLES}

    for x, y in edges:
        neighbors[x].add(y)
        neighbors[y].add(x)

    return neighbors


NEIGHBORS = build_neighbors(EDGES)


def is_consistent(variable, value, assignment):
    for neighbor in NEIGHBORS[variable]:
        if neighbor in assignment:
            if assignment[neighbor] == value:
                return False

    return True


class SearchStats:
    def __init__(self):
        self.nodes = 0


def basic_backtracking(assignment, domains, stats):
    if len(assignment) == len(VARIABLES):
        return assignment.copy()

    variable = next(
        v for v in VARIABLES
        if v not in assignment
    )

    for value in domains[variable]:
        stats.nodes += 1

        if is_consistent(variable, value, assignment):
            assignment[variable] = value

            result = basic_backtracking(
                assignment,
                domains,
                stats
            )

            if result is not None:
                return result

            del assignment[variable]

    return None


def forward_check(variable, value, assignment, domains):
    new_domains = deepcopy(domains)
    new_domains[variable] = [value]

    for neighbor in NEIGHBORS[variable]:
        if neighbor in assignment:
            continue

        if value in new_domains[neighbor]:
            new_domains[neighbor].remove(value)

        if len(new_domains[neighbor]) == 0:
            return None

    return new_domains


def revise(domains, x, y):
    revised = False

    for x_value in domains[x][:]:
        supported = any(
            x_value != y_value
            for y_value in domains[y]
        )

        if not supported:
            domains[x].remove(x_value)
            revised = True

    return revised


def ac3(domains):
    queue = deque()

    for x in VARIABLES:
        for y in NEIGHBORS[x]:
            queue.append((x, y))

    while queue:
        x, y = queue.popleft()

        if revise(domains, x, y):
            if len(domains[x]) == 0:
                return False

            for z in NEIGHBORS[x]:
                if z != y:
                    queue.append((z, x))

    return True


def select_mrv_variable(assignment, domains):
    unassigned = [
        v for v in VARIABLES
        if v not in assignment
    ]

    return min(
        unassigned,
        key=lambda v: len(domains[v])
    )


def improved_backtracking(assignment, domains, stats):
    if len(assignment) == len(VARIABLES):
        return assignment.copy()

    variable = select_mrv_variable(
        assignment,
        domains
    )

    for value in domains[variable]:
        stats.nodes += 1

        if not is_consistent(variable, value, assignment):
            continue

        new_domains = forward_check(
            variable,
            value,
            assignment,
            domains
        )

        if new_domains is None:
            continue

        if not ac3(new_domains):
            continue

        assignment[variable] = value

        result = improved_backtracking(
            assignment,
            new_domains,
            stats
        )

        if result is not None:
            return result

        del assignment[variable]

    return None


def main():
    print("=" * 60)
    print("CONSTRAINT SATISFACTION PROBLEM")
    print("Graph Coloring")
    print("=" * 60)

    print("\nVariables:")
    print(VARIABLES)

    print("\nColors:")
    print(COLORS)

    print("\nConstraints:")
    for x, y in EDGES:
        print(f"  {x} != {y}")

    basic_stats = SearchStats()

    basic_solution = basic_backtracking(
        assignment={},
        domains=deepcopy(DOMAINS),
        stats=basic_stats
    )

    improved_stats = SearchStats()

    improved_solution = improved_backtracking(
        assignment={},
        domains=deepcopy(DOMAINS),
        stats=improved_stats
    )

    print("\n" + "=" * 60)
    print("BASIC BACKTRACKING")
    print("=" * 60)

    print("Solution:")
    print(basic_solution)

    print("Nodes explored:",
          basic_stats.nodes)

    print("\n" + "=" * 60)
    print("IMPROVED BACKTRACKING")
    print("=" * 60)

    print("Solution:")
    print(improved_solution)

    print("Nodes explored:",
          improved_stats.nodes)

    print("\n" + "=" * 60)
    print("COMPARISON")
    print("=" * 60)

    if improved_stats.nodes < basic_stats.nodes:
        reduction = (
            100
            * (basic_stats.nodes - improved_stats.nodes)
            / basic_stats.nodes
        )

        print(
            f"Basic search explored:     {basic_stats.nodes}"
        )
        print(
            f"Improved search explored:  {improved_stats.nodes}"
        )
        print(
            f"Reduction in exploration:  {reduction:.1f}%"
        )

    elif improved_stats.nodes == basic_stats.nodes:
        print("Both methods explored the same number of nodes.")

    else:
        print(
            "The improved solver explored more nodes on "
            "this particular instance."
        )


if __name__ == "__main__":
    main()

CONSTRAINT SATISFACTION PROBLEM
Graph Coloring

Variables:
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

Colors:
['Red', 'Green', 'Blue', 'Yellow']

Constraints:
  A != B
  A != C
  A != D
  B != C
  B != E
  B != F
  C != D
  C != E
  C != F
  D != E
  D != G
  E != F
  E != G
  E != H
  F != G
  F != H
  G != H

BASIC BACKTRACKING
Solution:
{'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Green', 'E': 'Red', 'F': 'Yellow', 'G': 'Blue', 'H': 'Green'}
Nodes explored: 18

IMPROVED BACKTRACKING
Solution:
{'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Green', 'E': 'Red', 'F': 'Yellow', 'G': 'Blue', 'H': 'Green'}
Nodes explored: 8

COMPARISON
Basic search explored:     18
Improved search explored:  8
Reduction in exploration:  55.6%


In [ ]:
import heapq


def astar(grid, start, goal):

    rows = len(grid)
    cols = len(grid[0])

    # ======================================================
    # 1. HEURISTIC: h(n)
    # ======================================================
    # 4-direction movement হলে Manhattan distance ব্যবহার করি.
    # প্রতিটি move-এর minimum cost 1, তাই Manhattan distance
    # কখনো actual remaining cost-এর চেয়ে বেশি হবে না.
    def heuristic(node):
        r, c = node
        gr, gc = goal

        return abs(r - gr) + abs(c - gc)


    # ======================================================
    # 2. COST OF ENTERING A CELL
    # ======================================================
    def cost(cell):
        r, c = cell

        if grid[r][c] == '.':
            return 1

        if grid[r][c] == '~':
            return 3

        return float('inf')


    # ======================================================
    # 3. GET VALID NEIGHBORS
    # ======================================================
    def get_neighbors(node):

        r, c = node

        directions = [
            (-1, 0),   # UP
            (1, 0),    # DOWN
            (0, -1),   # LEFT
            (0, 1)     # RIGHT
        ]

        for dr, dc in directions:

            nr = r + dr
            nc = c + dc

            # Outside grid?
            if nr < 0 or nr >= rows:
                continue

            if nc < 0 or nc >= cols:
                continue

            # Wall?
            if grid[nr][nc] == '#':
                continue

            yield (nr, nc)


    # ======================================================
    # 4. g(n)
    # ======================================================
    # Cheapest known cost from START -> node
    g_score = {
        start: 0
    }


    # ======================================================
    # 5. came_from
    # ======================================================
    # Path reconstruct করার জন্য
    came_from = {}


    # ======================================================
    # 6. OPEN = MIN HEAP
    # ======================================================
    # Store:
    # (f, g, node)
    #
    # f = g + h
    # ======================================================
    open_set = []

    start_h = heuristic(start)

    heapq.heappush(
        open_set,
        (start_h, 0, start)
    )


    # ======================================================
    # 7. A* SEARCH
    # ======================================================
    while open_set:

        f, current_g, current = heapq.heappop(open_set)


        # --------------------------------------------------
        # STALE ENTRY
        # --------------------------------------------------
        # একই node আগে heap-এ ছিল কিন্তু পরে cheaper path
        # পাওয়া গেলে পুরোনো entry ignore করব.
        if current_g != g_score.get(current, float('inf')):
            continue


        # --------------------------------------------------
        # GOAL
        # --------------------------------------------------
        if current == goal:

            path = []

            node = current

            while node != start:

                path.append(node)

                node = came_from[node]

            path.append(start)

            path.reverse()

            return path, current_g


        # --------------------------------------------------
        # EXPAND NEIGHBORS
        # --------------------------------------------------
        for neighbor in get_neighbors(current):

            # Cost of entering neighbor
            move_cost = cost(neighbor)


            # New actual cost
            new_g = current_g + move_cost


            # ------------------------------------------------
            # CHEAPER PATH?
            # ------------------------------------------------
            if new_g < g_score.get(
                neighbor,
                float('inf')
            ):

                # Update g
                g_score[neighbor] = new_g

                # Update parent
                came_from[neighbor] = current


                # h(n)
                h = heuristic(neighbor)


                # f(n) = g(n) + h(n)
                new_f = new_g + h


                # Put into OPEN
                heapq.heappush(
                    open_set,
                    (new_f, new_g, neighbor)
                )


    # ======================================================
    # NO PATH
    # ======================================================
    return [], -1